# Notebook 02: Feature Engineering & Data Preparation

**Goal**: Clean the training data, engineer features, and prepare
train/validation/test splits for the AVM.

**Key decisions**:
- Focus on single-family residential (classes 202-210) for the core AVM
- Include multi-family (211, 212) and townhomes (210, 295) for broader coverage
- Use the same feature set as CCAO where possible, but in Python/sklearn

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

In [2]:
# Load data from Notebook 01
training = pd.read_parquet("../data/raw/training_data.parquet")
assessment = pd.read_parquet("../data/raw/assessment_data.parquet")
land_rates = pd.read_parquet("../data/raw/land_nbhd_rate_data.parquet")
print(f"Training: {training.shape}")
print(f"Assessment: {assessment.shape}")

Training: (413289, 201)
Assessment: (1100173, 191)


## 1. Identify Target & Key Columns

The CCAO model predicts sale price. We need to confirm the exact column
name and understand the data schema.

In [5]:
TARGET_COL = "meta_sale_price"

print(f"Target: {TARGET_COL}")
print(f"  Non-null: {training[TARGET_COL].notna().sum():,}")
print(f"  Range: ${training[TARGET_COL].min():,.0f} - ${training[TARGET_COL].max():,.0f}")

Target: meta_sale_price
  Non-null: 413,289
  Range: $10,001 - $16,000,000


## 2. Filter Property Classes

In [7]:
# Identify the class column
CLASS_COL = "meta_class"
print(f"Using class column: {CLASS_COL}")

# Single-family + multi-family residential classes
# These are the classes the CCAO's model-res-avm handles
RESIDENTIAL_CLASSES = [
    "202", "203", "204", "205", "206", "207", "208", "209",  # Single-family
    "210", "234", "278", "295",  # Townhouse / split-use / coach house
    "211", "212",  # Multi-family (2-6 units)
]

# Convert class to string for consistent matching
training[CLASS_COL] = training[CLASS_COL].astype(str).str.strip()
assessment[CLASS_COL] = assessment[CLASS_COL].astype(str).str.strip()

train_res = training[training[CLASS_COL].isin(RESIDENTIAL_CLASSES)].copy()
assess_res = assessment[assessment[CLASS_COL].isin(RESIDENTIAL_CLASSES)].copy()

print(f"Training (all): {len(training):,}")
print(f"Training (residential): {len(train_res):,} ({100*len(train_res)/len(training):.1f}%)")
print(f"Assessment (residential): {len(assess_res):,}")

Using class column: meta_class
Training (all): 413,289
Training (residential): 413,282 (100.0%)
Assessment (residential): 1,100,150


## 3. Remove Outliers in Sale Price

In [8]:
# The CCAO already filters for arms-length sales, but we should still
# trim extreme outliers that could hurt model performance.
print("=== Sale Price Outlier Removal ===")
q01 = train_res[TARGET_COL].quantile(0.01)
q99 = train_res[TARGET_COL].quantile(0.99)
print(f"  1st percentile: ${q01:,.0f}")
print(f"  99th percentile: ${q99:,.0f}")

before = len(train_res)
train_res = train_res[
    (train_res[TARGET_COL] >= q01) & 
    (train_res[TARGET_COL] <= q99) &
    (train_res[TARGET_COL] > 0)
].copy()
after = len(train_res)
print(f"  Removed {before - after:,} outliers ({100*(before-after)/before:.1f}%)")
print(f"  Remaining: {after:,} sales")

=== Sale Price Outlier Removal ===
  1st percentile: $23,000
  99th percentile: $1,775,000
  Removed 8,172 outliers (2.0%)
  Remaining: 405,110 sales


## 4. Define Feature Groups

In [9]:
# === Organize columns into feature groups ===
# Mirrors the CCAO's feature categories from their README

# Characteristic features (property-level)
CHAR_NUMERIC = [
    "char_land_sf", "char_bldg_sf", "char_yrblt", "char_beds",
    "char_rooms", "char_fbath", "char_hbath", "char_frpl", "char_ncu",
]

CHAR_CATEGORICAL = [
    "char_air", "char_apts", "char_attic_fnsh", "char_attic_type",
    "char_bsmt", "char_bsmt_fin", "char_ext_wall", "char_gar1_att",
    "char_gar1_cnst", "char_gar1_size", "char_heat", "char_porch",
    "char_roof_cnst", "char_tp_dsgn", "char_type_resd",
]

# Location features
LOC_NUMERIC = [
    "loc_latitude", "loc_longitude",
    "loc_access_cmap_walk_total_score", "loc_access_cmap_walk_nta_score",
    "loc_env_flood_fs_factor",
]

LOC_CATEGORICAL = [
    "loc_census_tract_geoid",
    "loc_school_elementary_district_geoid",
    "loc_school_secondary_district_geoid",
    "loc_tax_municipality_name",
]

# Proximity features
PROX_NUMERIC = [c for c in train_res.columns if c.startswith("prox_")]

# ACS/Census features
ACS_NUMERIC = [c for c in train_res.columns if c.startswith("acs5_")]

# Time features (for training only — assessment uses a fixed date)
TIME_FEATURES = [c for c in train_res.columns if c.startswith("time_")]

# Parcel shape features
SHAPE_FEATURES = [c for c in train_res.columns if c.startswith("shp_")]

# Other
OTHER_FEATURES = ["other_tax_bill_rate"] if "other_tax_bill_rate" in train_res.columns else []

# Meta features (identifiers, not model inputs, but needed for joins)
META_FEATURES = ["meta_pin", "meta_nbhd_code", "meta_township_code", "meta_card_num"]

# Filter to columns that actually exist in the data
def filter_existing(col_list, df):
    return [c for c in col_list if c in df.columns]

CHAR_NUMERIC = filter_existing(CHAR_NUMERIC, train_res)
CHAR_CATEGORICAL = filter_existing(CHAR_CATEGORICAL, train_res)
LOC_NUMERIC = filter_existing(LOC_NUMERIC, train_res)
LOC_CATEGORICAL = filter_existing(LOC_CATEGORICAL, train_res)
PROX_NUMERIC = filter_existing(PROX_NUMERIC, train_res)
ACS_NUMERIC = filter_existing(ACS_NUMERIC, train_res)
TIME_FEATURES = filter_existing(TIME_FEATURES, train_res)
SHAPE_FEATURES = filter_existing(SHAPE_FEATURES, train_res)
OTHER_FEATURES = filter_existing(OTHER_FEATURES, train_res)
META_FEATURES = filter_existing(META_FEATURES, train_res)

ALL_NUMERIC = CHAR_NUMERIC + LOC_NUMERIC + PROX_NUMERIC + ACS_NUMERIC + SHAPE_FEATURES + OTHER_FEATURES
ALL_CATEGORICAL = CHAR_CATEGORICAL + LOC_CATEGORICAL

print(f"Numeric features:     {len(ALL_NUMERIC)}")
print(f"Categorical features: {len(ALL_CATEGORICAL)}")
print(f"Time features:        {len(TIME_FEATURES)}")
print(f"Total features:       {len(ALL_NUMERIC) + len(ALL_CATEGORICAL) + len(TIME_FEATURES)}")

Numeric features:     84
Categorical features: 19
Time features:        8
Total features:       111


## 5. Engineer New Features

In [10]:
def engineer_features(df):
    """Add derived features relevant to the land-improvement decomposition."""
    df = df.copy()
    
    # === Age-related ===
    # Use the sale year if available, otherwise use 2025
    if "time_sale_year" in df.columns:
        ref_year = df["time_sale_year"]
    else:
        ref_year = 2025
    
    if "char_yrblt" in df.columns:
        df["feat_age"] = ref_year - df["char_yrblt"]
        df["feat_age"] = df["feat_age"].clip(lower=0, upper=200)
        df["feat_age_squared"] = df["feat_age"] ** 2
    
    # === Size ratios (CRITICAL for land-improvement decomposition) ===
    if "char_bldg_sf" in df.columns and "char_land_sf" in df.columns:
        # Building-to-land ratio: high ratio = dense improvement
        df["feat_bldg_to_land_ratio"] = (
            df["char_bldg_sf"] / df["char_land_sf"].replace(0, np.nan)
        )
        df["feat_bldg_to_land_ratio"] = df["feat_bldg_to_land_ratio"].clip(upper=5)
        
        # Floor area ratio (FAR) — proxy for intensity of use
        # Rough: bldg_sf / land_sf (already captured above, but conceptually useful)
        
        # Log transforms for skewed size features
        df["feat_log_land_sf"] = np.log1p(df["char_land_sf"])
        df["feat_log_bldg_sf"] = np.log1p(df["char_bldg_sf"])
    
    # === Total bathroom count ===
    if "char_fbath" in df.columns and "char_hbath" in df.columns:
        df["feat_total_baths"] = df["char_fbath"] + 0.5 * df["char_hbath"]
    
    # === Room density (rooms per sqft) ===
    if "char_rooms" in df.columns and "char_bldg_sf" in df.columns:
        df["feat_rooms_per_sf"] = (
            df["char_rooms"] / df["char_bldg_sf"].replace(0, np.nan)
        )
    
    # === Land value indicators ===
    if "char_land_sf" in df.columns:
        df["feat_log_land_sf"] = np.log1p(df["char_land_sf"])
        # Large lot indicator (above 75th percentile)
        q75 = df["char_land_sf"].quantile(0.75)
        df["feat_large_lot"] = (df["char_land_sf"] > q75).astype(int)
    
    return df

# Apply to both datasets
train_res = engineer_features(train_res)
assess_res = engineer_features(assess_res)

# Update feature lists
ENGINEERED_NUMERIC = [c for c in train_res.columns if c.startswith("feat_")]
ALL_NUMERIC_FINAL = ALL_NUMERIC + ENGINEERED_NUMERIC

print(f"Engineered features: {ENGINEERED_NUMERIC}")
print(f"Total numeric features: {len(ALL_NUMERIC_FINAL)}")

Engineered features: ['feat_age', 'feat_age_squared', 'feat_bldg_to_land_ratio', 'feat_log_land_sf', 'feat_log_bldg_sf', 'feat_total_baths', 'feat_rooms_per_sf', 'feat_large_lot']
Total numeric features: 92


## 6. Handle Missing Values

In [11]:
# Check missingness in feature columns
feature_cols = ALL_NUMERIC_FINAL + ALL_CATEGORICAL + TIME_FEATURES
missing = train_res[feature_cols].isna().mean().sort_values(ascending=False)
missing_any = missing[missing > 0]

print(f"=== Missing Values in Feature Columns ===")
print(f"Columns with missing data: {len(missing_any)} / {len(feature_cols)}")
print()
for col, pct in missing_any.items():
    print(f"  {col:<55} {pct:.1%}")

=== Missing Values in Feature Columns ===
Columns with missing data: 106 / 119

  char_tp_dsgn                                            52.5%
  prox_nearest_road_highway_speed_limit                   43.6%
  prox_avg_school_rating_in_half_mile                     25.2%
  prox_nearest_road_collector_speed_limit                 17.3%
  char_gar1_cnst                                          15.5%
  acs5_median_household_total_occupied_year_built         11.2%
  prox_nearest_road_highway_daily_traffic                 10.5%
  acs5_median_household_renter_occupied_gross_rent        3.3%
  prox_nearest_road_collector_daily_traffic               1.0%
  prox_nearest_golf_course_dist_ft                        0.7%
  loc_env_flood_fs_factor                                 0.4%
  char_frpl                                               0.4%
  acs5_median_household_owner_occupied_value              0.3%
  prox_nearest_university_dist_ft                         0.3%
  prox_nearest_hospital_dist_ft

In [12]:
# === Imputation Strategy ===
# LightGBM can handle NaN natively, so we have two options:
#   1. Let LightGBM handle it (simplest, and what CCAO does)
#   2. Impute explicitly (needed for sklearn models like XGBoost < 2.0)
#
# We'll do option 1 for LightGBM, but prepare a simple imputer for XGBoost baseline

from sklearn.impute import SimpleImputer

# For numeric: median imputation
num_imputer = SimpleImputer(strategy="median")
# For categorical: most frequent (or "missing" category)
cat_imputer = SimpleImputer(strategy="most_frequent")

# We'll apply these in the model training notebook as part of a pipeline
# For now, just note the strategy
print("Imputation strategy:")
print("  Numeric \u2192 median (or LightGBM native NaN handling)")
print("  Categorical \u2192 most frequent value")

Imputation strategy:
  Numeric → median (or LightGBM native NaN handling)
  Categorical → most frequent value


## 7. Encode Categorical Variables

In [13]:
# LightGBM handles categoricals natively (one of its key advantages).
# We need to convert them to pandas Categorical type.
# For XGBoost, we'll use ordinal encoding.

from sklearn.preprocessing import OrdinalEncoder

# Convert categoricals for LightGBM (native handling)
for col in ALL_CATEGORICAL:
    if col in train_res.columns:
        train_res[col] = train_res[col].astype("category")
    if col in assess_res.columns:
        assess_res[col] = assess_res[col].astype("category")

print("Categorical columns converted to pandas Categorical dtype.")
print(f"Unique categories per column:")
for col in ALL_CATEGORICAL[:10]:  # Show first 10
    if col in train_res.columns:
        n = train_res[col].nunique()
        print(f"  {col}: {n} categories")

Categorical columns converted to pandas Categorical dtype.
Unique categories per column:
  char_air: 2 categories
  char_apts: 7 categories
  char_attic_fnsh: 3 categories
  char_attic_type: 3 categories
  char_bsmt: 4 categories
  char_bsmt_fin: 3 categories
  char_ext_wall: 4 categories
  char_gar1_att: 2 categories
  char_gar1_cnst: 4 categories
  char_gar1_size: 8 categories


## 8. Time-Based Train/Validation/Test Split

Following CCAO's approach: out-of-time testing.
The most recent 10% of sales are the test set.
The remaining 90% is split 80/20 into train/validation.

In [14]:
# Identify the time column for splitting
TIME_COL = None
for candidate in ["time_sale_day", "time_sale_year", "meta_sale_date"]:
    if candidate in train_res.columns:
        TIME_COL = candidate
        break

print(f"Splitting on: {TIME_COL}")

if TIME_COL:
    # Sort by time
    train_res = train_res.sort_values(TIME_COL).reset_index(drop=True)
    
    n = len(train_res)
    test_cutoff = int(n * 0.90)
    val_cutoff = int(n * 0.80)
    
    df_train = train_res.iloc[:val_cutoff].copy()
    df_val = train_res.iloc[val_cutoff:test_cutoff].copy()
    df_test = train_res.iloc[test_cutoff:].copy()
    
    print(f"Train:      {len(df_train):>8,} ({100*len(df_train)/n:.1f}%)")
    print(f"Validation: {len(df_val):>8,} ({100*len(df_val)/n:.1f}%)")
    print(f"Test:       {len(df_test):>8,} ({100*len(df_test)/n:.1f}%)")
    
    if "time_sale_year" in train_res.columns:
        print(f"\nTrain years: {df_train['time_sale_year'].min()}-{df_train['time_sale_year'].max()}")
        print(f"Val years:   {df_val['time_sale_year'].min()}-{df_val['time_sale_year'].max()}")
        print(f"Test years:  {df_test['time_sale_year'].min()}-{df_test['time_sale_year'].max()}")
else:
    # Fall back to random split
    from sklearn.model_selection import train_test_split
    df_trainval, df_test = train_test_split(train_res, test_size=0.1, random_state=42)
    df_train, df_val = train_test_split(df_trainval, test_size=0.11, random_state=42)  # ~10% of total
    print(f"Train:      {len(df_train):>8,}")
    print(f"Validation: {len(df_val):>8,}")
    print(f"Test:       {len(df_test):>8,}")

Splitting on: time_sale_day
Train:       324,088 (80.0%)
Validation:   40,511 (10.0%)
Test:         40,511 (10.0%)

Train years: 2016.0-2022.0
Val years:   2022.0-2023.0
Test years:  2023.0-2024.0


## 9. Prepare Feature Matrices

In [15]:
# === Define final feature set ===
# We include time features for training but NOT for assessment prediction
# (assessment uses a fixed date, so time features are set to Jan 1 of assessment year)

FEATURES_FOR_MODEL = ALL_NUMERIC_FINAL + ALL_CATEGORICAL
TIME_FEATURES_FOR_MODEL = [c for c in TIME_FEATURES if c in train_res.columns]

# For training, include time features
TRAIN_FEATURES = FEATURES_FOR_MODEL + TIME_FEATURES_FOR_MODEL

# Filter to columns that exist
TRAIN_FEATURES = [c for c in TRAIN_FEATURES if c in df_train.columns]

print(f"=== Final Feature Set ===")
print(f"Total features for training: {len(TRAIN_FEATURES)}")
print(f"  Numeric (char):     {len(CHAR_NUMERIC)}")
print(f"  Numeric (loc):      {len(LOC_NUMERIC)}")
print(f"  Numeric (prox):     {len(PROX_NUMERIC)}")
print(f"  Numeric (acs):      {len(ACS_NUMERIC)}")
print(f"  Numeric (shape):    {len(SHAPE_FEATURES)}")
print(f"  Numeric (other):    {len(OTHER_FEATURES)}")
print(f"  Numeric (engineered): {len(ENGINEERED_NUMERIC)}")
print(f"  Categorical:        {len(ALL_CATEGORICAL)}")
print(f"  Time:               {len(TIME_FEATURES_FOR_MODEL)}")

=== Final Feature Set ===
Total features for training: 119
  Numeric (char):     9
  Numeric (loc):      5
  Numeric (prox):     41
  Numeric (acs):      22
  Numeric (shape):    6
  Numeric (other):    1
  Numeric (engineered): 8
  Categorical:        19
  Time:               8


In [16]:
# Build X, y matrices
X_train = df_train[TRAIN_FEATURES]
y_train = df_train[TARGET_COL]

X_val = df_val[TRAIN_FEATURES]
y_val = df_val[TARGET_COL]

X_test = df_test[TRAIN_FEATURES]
y_test = df_test[TARGET_COL]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

X_train: (324088, 119), y_train: (324088,)
X_val:   (40511, 119), y_val:   (40511,)
X_test:  (40511, 119), y_test:  (40511,)


## 10. Save Processed Data

In [18]:
# Save splits for use in notebook 03
df_train.to_parquet("../data/processed/train_split.parquet", index=False)
df_val.to_parquet("../data/processed/val_split.parquet", index=False)
df_test.to_parquet("../data/processed/test_split.parquet", index=False)
assess_res.to_parquet("../data/processed/assessment_clean.parquet", index=False)

# Save feature lists as JSON for consistency across notebooks
import json

feature_config = {
    "target_col": TARGET_COL,
    "class_col": CLASS_COL,
    "train_features": TRAIN_FEATURES,
    "all_numeric": ALL_NUMERIC_FINAL,
    "all_categorical": ALL_CATEGORICAL,
    "time_features": TIME_FEATURES_FOR_MODEL,
    "char_numeric": CHAR_NUMERIC,
    "engineered_numeric": ENGINEERED_NUMERIC,
    "residential_classes": RESIDENTIAL_CLASSES,
}

with open("../data/processed/feature_config.json", "w") as f:
    json.dump(feature_config, f, indent=2)

print("Saved:")
print("  ../data/processed/train_split.parquet")
print("  ../data/processed/val_split.parquet")
print("  ../data/processed/test_split.parquet")
print("  ../data/processed/assessment_clean.parquet")
print("  ../data/processed/feature_config.json")

Saved:
  ../data/processed/train_split.parquet
  ../data/processed/val_split.parquet
  ../data/processed/test_split.parquet
  ../data/processed/assessment_clean.parquet
  ../data/processed/feature_config.json
